#1. Preprocessing

This preprocessing block creates the first harmonised dataset used throughout the analysis. At this stage, the objective is not yet to build the final modelling matrix, but to convert all raw signal files into a standard acquisition-level representation that is compatible with both the old-code and new-code measurement protocols. Each signal file is treated as one acquisition, and each acquisition is described by a common set of metadata fields, including protocol version, patient identifier, anatomical site, measurement index within site, source filename, and the set of available frequency channels.

A key design decision of the workflow is that raw frequencies are not forced into a single wide table at this stage. The old-code and new-code protocols differ in the number of repeated measurements per site and may also differ in their frequency grids. Instead of creating a sparse frequency matrix full of missing values, the raw signal content is stored per acquisition as a dictionary mapping frequency to signal array. In parallel, a long-format table is also created with one row per acquisition and frequency channel. This preserves the original information while keeping the preprocessing transparent and flexible for later feature extraction.

For the purposes of this thesis, the new-code filename structure is simplified analytically. Although filenames contain angle and repetition markers, these are not retained as separate scientific variables because they do not represent distinct acquisition geometries in the present application. They are used only to generate a stable measurement index from 1 to 6 within anatomical site. In old code, the equivalent index runs from 1 to 3. The resulting dataset therefore uses a common acquisition structure across both protocols while preserving patient identifiers and source-level provenance.

In [3]:
# This block builds the harmonised source-level dataset used in the analysis.
# It integrates old-code and new-code signal files, elastograph metadata, and
# clinical tables. Each signal file is treated as one acquisition. At this
# stage, frequencies are kept as dictionaries {frequency_hz: np.array} rather
# than being forced into a single sparse wide matrix.
#
# Main design choices
# - one row = one signal file
# - old code and new code share the same metadata schema
# - raw frequencies are preserved as dictionaries
# - new-code angle and repetition are collapsed into measurement_index_site = 1..6
# - old-code repetitions remain measurement_index_site = 1..3
# - timepoint is attached afterwards using elastograph metadata

import re
import time
import hashlib
from pathlib import Path
from typing import Optional, List, Dict

import numpy as np
import pandas as pd
from IPython.display import display
from google.colab import drive

## 1.1. Configuration

In [4]:
drive.mount("/content/drive")

# New-code local IDs 1..86 correspond to final patient IDs 159..244
NEW_PATIENT_OFFSET = 158

SITE_LABEL_MAP = {
    "A": "A",
    "B": "B",
    "P": "P",
}

TIMEPOINT_ORDER = {
    "PRE": 0,
    "POST": 1,
    "FUP": 2,
    "UNKNOWN": 99,
}

EXPECTED_MEASUREMENTS_PER_SITE = {
    "old": 3,
    "new": 6,
}

EXPECTED_SITES = ["A", "B", "P"]

EXPECTED_ACQUISITIONS_PER_VISIT = {
    protocol: n_rep * len(EXPECTED_SITES)
    for protocol, n_rep in EXPECTED_MEASUREMENTS_PER_SITE.items()
}


Mounted at /content/drive


## 1.2. Local functions


In [5]:
def sha256_file(file_path: str, chunk_size: int = 1 << 20) -> str:
    """
    Compute a SHA256 hash for provenance tracking.
    Disabled by default because it can slow down preprocessing.
    """
    h = hashlib.sha256()
    with open(file_path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

In [6]:
def clean_text(x) -> str:
    """Remove stray quotes and surrounding whitespace."""
    return str(x).replace('"', "").strip()

In [7]:
def extract_digits(x) -> Optional[int]:
    """Extract the first integer found in a string-like object."""
    m = re.search(r"(\d+)", clean_text(x))
    return int(m.group(1)) if m else None

In [8]:
def parse_timepoint_from_text(x) -> str:
    """
    Canonical timepoint mapping.
    Accepts PRE / POST / FOLLOW-UP / FUP-like values.
    """
    s = clean_text(x).upper()
    if "POST" in s:
        return "POST"
    if "PRE" in s:
        return "PRE"
    if "FUP" in s or "FOLLOW" in s or "_FU" in s:
        return "FUP"
    return "UNKNOWN"

In [9]:
def parse_timepoint_from_filename(file_path: str) -> str:
    """Infer timepoint from a filename when files are stored per visit."""
    return parse_timepoint_from_text(Path(file_path).name)

In [10]:
def sort_timepoints(values: List[str]) -> List[str]:
    """Sort canonical timepoints in study order."""
    return sorted(values, key=lambda x: TIMEPOINT_ORDER.get(x, 99))

In [11]:
def canonical_site_label(site_code: str) -> str:
    """
    Map A/B/P to a canonical label.
    Neutral mapping is kept at this stage to avoid imposing a wrong anatomy label.
    """
    site_code = clean_text(site_code).upper()
    return SITE_LABEL_MAP.get(site_code, site_code)

In [12]:
def extract_frequency_hz(col_name) -> Optional[int]:
    """
    Extract a frequency in Hz from a column name such as 'F1_700Hz' or '700 Hz'.
    """
    m = re.search(r"(\d+)\s*Hz", str(col_name), flags=re.IGNORECASE)
    return int(m.group(1)) if m else None

In [13]:
def infer_protocol_from_filename(file_path: str) -> str:
    """
    Infer whether a signal file belongs to old code or new code.
    """
    stem = Path(file_path).stem
    if re.search(r"angle_\d+_rep_\d+", stem, flags=re.IGNORECASE):
        return "new"
    return "old"

In [14]:
def parse_datetime_token(token: str) -> pd.Timestamp:
    """
    Parse tokens such as 03_02_24_08_43_24 or 23_03_25_09_48_52.
    Expected format: dd_mm_yy_HH_MM_SS
    """
    token = clean_text(token)
    return pd.to_datetime(token, format="%d_%m_%y_%H_%M_%S", errors="coerce")


## 1.3. Signal filename parsers

In [15]:
def parse_oldcode_filename(file_path: str) -> dict:
    """
    Parse an old-code signal filename.

    Example
        in_test_0001_A103_02_24_08_43_24.csv

    Interpretation
    - local_patient_id = 0001
    - global_patient_id = 0001
    - site_code = A
    - measurement_index_site = 1
    - remaining token is used as acquisition token
    """
    stem = Path(file_path).stem
    parts = stem.split("_")

    if len(parts) < 4:
        raise ValueError(f"Unexpected old-code filename: {Path(file_path).name}")

    local_patient_id = extract_digits(parts[2])
    probe = parts[3]

    if local_patient_id is None or len(probe) < 2:
        raise ValueError(f"Could not parse old-code filename: {Path(file_path).name}")

    site_code = canonical_site_label(probe[0])

    try:
        measurement_index_site = int(probe[1])
    except Exception:
        raise ValueError(f"Could not parse old-code measurement index from: {Path(file_path).name}")

    acquisition_token = "_".join([probe[2:]] + parts[4:]).strip("_")
    acquisition_datetime = parse_datetime_token(acquisition_token)

    return {
        "protocol_version": "old",
        "local_patient_id": int(local_patient_id),
        "global_patient_id": int(local_patient_id),
        "site_code": site_code,
        "measurement_index_site": measurement_index_site,
        "acquisition_token": acquisition_token,
        "acquisition_datetime": acquisition_datetime,
    }


In [16]:
def parse_newcode_filename(
    file_path: str,
    new_patient_offset: int = NEW_PATIENT_OFFSET
) -> dict:
    """
    Parse a new-code signal filename.

    Example
        in_test_0001_A_Study_angle_1_rep_1_23_03_25_09_48_52.csv

    For this thesis
    - angle and repetition are not kept as scientific variables
    - they are only used to define measurement_index_site = 1..6
    """
    stem = Path(file_path).stem

    m = re.match(
        r"^in_test_(\d+)_([ABP])_.*?angle_(\d+)_rep_(\d+)_(.+)$",
        stem,
        flags=re.IGNORECASE,
    )
    if m is None:
        raise ValueError(f"Unexpected new-code filename: {Path(file_path).name}")

    local_patient_id = int(m.group(1))
    site_code = canonical_site_label(m.group(2))
    angle = int(m.group(3))
    repetition = int(m.group(4))
    acquisition_token = m.group(5)

    measurement_index_site = (angle - 1) * 2 + repetition
    acquisition_datetime = parse_datetime_token(acquisition_token)

    return {
        "protocol_version": "new",
        "local_patient_id": local_patient_id,
        "global_patient_id": local_patient_id + new_patient_offset,
        "site_code": site_code,
        "measurement_index_site": measurement_index_site,
        "acquisition_token": acquisition_token,
        "acquisition_datetime": acquisition_datetime,
    }

In [17]:
def parse_signal_filename(file_path: str) -> dict:
    """Dispatch to the correct parser based on the filename pattern."""
    protocol = infer_protocol_from_filename(file_path)
    if protocol == "new":
        return parse_newcode_filename(file_path)
    return parse_oldcode_filename(file_path)

##1.4. Signal file  loading

In [18]:
def read_signal_csv(file_path: str) -> pd.DataFrame:
    """
    Read one raw signal CSV file.
    """
    return pd.read_csv(file_path)

In [19]:
def load_signal_file_record(file_path: str, compute_hash: bool = False) -> dict:
    """
    Convert one raw signal file into one harmonised acquisition-level record.

    Frequencies are stored as a dictionary {frequency_hz: np.array}.
    """
    meta = parse_signal_filename(file_path)
    df = read_signal_csv(file_path)

    freq_to_array = {}
    n_points_by_freq = {}

    for col in df.columns:
        freq_hz = extract_frequency_hz(col)
        if freq_hz is None:
            continue

        arr = pd.to_numeric(df[col], errors="coerce").dropna().to_numpy(dtype=np.float32)
        if arr.size == 0:
            continue

        freq_to_array[int(freq_hz)] = arr
        n_points_by_freq[int(freq_hz)] = int(arr.size)

    available_freqs = sorted(freq_to_array.keys())
    acquisition_date = pd.NaT
    if pd.notna(meta["acquisition_datetime"]):
        acquisition_date = pd.Timestamp(meta["acquisition_datetime"]).normalize()

    out = {
        "source_file": Path(file_path).name,
        "source_path": str(file_path),
        **meta,
        "acquisition_date": acquisition_date,
        "n_frequency_channels": len(available_freqs),
        "available_freqs_hz": available_freqs,
        "n_points_by_freq": n_points_by_freq,
        "freq_to_array": freq_to_array,
        "parse_ok": len(available_freqs) > 0,
    }

    if compute_hash:
        out["file_hash"] = sha256_file(file_path)

    return out

In [20]:
def collect_signal_files(signal_roots: List[str], recursive: bool = True) -> List[str]:
    """
    Collect signal CSV files from one or more directories.
    Only files starting with in_test_ are included.
    """
    files = []
    pattern = "**/in_test_*.csv" if recursive else "in_test_*.csv"

    for root in signal_roots:
        root_path = Path(root)
        if not root_path.exists():
            print(f"Warning: signal root does not exist -> {root}")
            continue

        found = list(root_path.glob(pattern))
        print(f"Signal root {root} -> {len(found)} files found")
        files.extend(str(p) for p in found)

    return sorted(set(files))

In [21]:
def build_signal_raw_dataset(signal_roots: List[str], compute_hash: bool = False) -> pd.DataFrame:
    """
    Build the acquisition-level raw signal dataset.

    Output
    - one row per file
    - protocol-aware metadata
    - raw frequency dictionary
    """
    signal_files = collect_signal_files(signal_roots=signal_roots, recursive=True)

    if len(signal_files) == 0:
        raise FileNotFoundError(
            "No signal files were found. Check your roots and filename pattern."
        )

    print(f"Signal files found in total: {len(signal_files)}")

    records = []
    errors = []

    t0 = time.time()
    for i, fp in enumerate(signal_files, start=1):
        if i % 50 == 0 or i == 1 or i == len(signal_files):
            print(f"Parsing file {i}/{len(signal_files)}: {Path(fp).name}")

        try:
            records.append(load_signal_file_record(fp, compute_hash=compute_hash))
        except Exception as e:
            errors.append({"source_file": Path(fp).name, "error": str(e)})

    t1 = time.time()
    print(f"Raw signal parsing finished in {t1 - t0:.2f} s")

    if errors:
        print("Warning: some files could not be parsed.")
        display(pd.DataFrame(errors).head(20))

    signals_raw_df = pd.DataFrame(records).reset_index(drop=True)
    signals_raw_df["signal_record_id"] = "sig_" + signals_raw_df.index.astype(str).str.zfill(6)

    ordered_cols = [
        "signal_record_id",
        "protocol_version",
        "local_patient_id",
        "global_patient_id",
        "site_code",
        "measurement_index_site",
        "source_file",
        "source_path",
        "acquisition_token",
        "acquisition_datetime",
        "acquisition_date",
        "n_frequency_channels",
        "available_freqs_hz",
        "parse_ok",
        "n_points_by_freq",
        "freq_to_array",
    ]
    if "file_hash" in signals_raw_df.columns:
        ordered_cols.append("file_hash")

    return signals_raw_df[[c for c in ordered_cols if c in signals_raw_df.columns]]

In [22]:
def build_signal_long_dataset_light(signals_df: pd.DataFrame) -> pd.DataFrame:
    """
    Build a light long-format signal table:
    one row = one acquisition x one frequency channel.
    The signal array itself is not duplicated here.
    """
    rows = []

    for row in signals_df.itertuples(index=False):
        if not isinstance(row.freq_to_array, dict):
            continue

        for freq_hz, arr in row.freq_to_array.items():
            out = {
                "signal_record_id": row.signal_record_id,
                "protocol_version": row.protocol_version,
                "global_patient_id": row.global_patient_id,
                "site_code": row.site_code,
                "measurement_index_site": row.measurement_index_site,
                "frequency_hz": int(freq_hz),
                "n_points": int(len(arr)),
            }

            if hasattr(row, "timepoint"):
                out["timepoint"] = row.timepoint
            if hasattr(row, "hospital_patient_id"):
                out["hospital_patient_id"] = row.hospital_patient_id
            if hasattr(row, "assignment_method"):
                out["assignment_method"] = row.assignment_method

            rows.append(out)

    return pd.DataFrame(rows)

## 1.5. Elastograph metadata

In [23]:
def clean_quoted_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    """
    Remove stray quotes and surrounding whitespace from all column names and values.
    """
    out = df.copy()
    out.columns = [clean_text(c) for c in out.columns]
    for c in out.columns:
        out[c] = out[c].map(clean_text)
    return out

In [24]:
def load_elastograph_metadata(
    csv_path: str,
    protocol_version: str,
    new_patient_offset: int = NEW_PATIENT_OFFSET,
) -> pd.DataFrame:
    """
    Load and standardise an elastograph metadata CSV.

    Canonical output fields
    - protocol_version
    - global_patient_id
    - hospital_patient_id
    - timepoint
    """
    try:
        df = pd.read_csv(csv_path, sep=',"', engine="python")
    except Exception:
        df = pd.read_csv(csv_path)

    df = clean_quoted_dataframe(df)

    required = {"PACIENTE", "ID", "Time"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing expected columns in {Path(csv_path).name}: {sorted(missing)}")

    df["global_patient_id"] = df["PACIENTE"].map(extract_digits).astype("Int64")

    # Apply offset if new-code metadata still stores local IDs 1..86
    if protocol_version == "new":
        max_pid = df["global_patient_id"].dropna().max()
        if pd.notna(max_pid) and max_pid <= 86:
            df["global_patient_id"] = df["global_patient_id"] + new_patient_offset

    df["hospital_patient_id"] = df["ID"].map(extract_digits).astype("Int64")
    df["timepoint"] = df["Time"].map(parse_timepoint_from_text)
    df["protocol_version"] = protocol_version

    out = df[
        ["protocol_version", "global_patient_id", "hospital_patient_id", "timepoint"]
    ].dropna(subset=["global_patient_id"]).copy()

    return out

In [25]:
def build_elasto_visit_table(elasto_df: pd.DataFrame) -> pd.DataFrame:
    """
    Reduce elastograph metadata to one row per patient x timepoint x protocol.
    """
    if elasto_df.empty:
        return pd.DataFrame(columns=[
            "protocol_version",
            "global_patient_id",
            "timepoint",
            "hospital_patient_id",
            "n_elasto_rows",
            "timepoint_order",
        ])

    tmp = elasto_df.copy()
    tmp["timepoint_order"] = tmp["timepoint"].map(TIMEPOINT_ORDER).fillna(99).astype(int)

    visit_df = (
        tmp.groupby(["protocol_version", "global_patient_id", "timepoint"], as_index=False)
        .agg(
            hospital_patient_id=("hospital_patient_id", lambda s: s.dropna().iloc[0] if s.dropna().shape[0] else pd.NA),
            n_elasto_rows=("hospital_patient_id", "size"),
        )
    )

    visit_df["timepoint_order"] = visit_df["timepoint"].map(TIMEPOINT_ORDER).fillna(99).astype(int)

    visit_df = visit_df.sort_values(
        ["protocol_version", "global_patient_id", "timepoint_order"]
    ).reset_index(drop=True)

    return visit_df

## 1.6. Clinical tables

In [26]:
def strip_timepoint_suffix_from_columns(df: pd.DataFrame, timepoint: str) -> pd.DataFrame:
    """
    Remove PRE / POST / FUP suffixes from variable names.

    Example
        BLOB_Gluc_PRE -> BLOB_Gluc
    """
    out = df.copy()
    tp = clean_text(timepoint).upper()

    def _strip(col: str) -> str:
        col = str(col).strip()
        suffix = f"_{tp}"
        if col.upper().endswith(suffix):
            return col[:-len(suffix)]
        return col

    out.columns = [_strip(c) for c in out.columns]
    return out

In [27]:
def load_clinical_files_to_long(
    clinical_files: List[str],
    id_col: str = "ID",
    static_cols: Optional[List[str]] = None,
    keep_base_vars: Optional[List[str]] = None,
) -> pd.DataFrame:
    """
    Convert one-file-per-timepoint clinical tables into a long dataset:
    one row = one hospital_patient_id x one timepoint
    """
    if static_cols is None:
        static_cols = ["Sex", "Age"]

    merged = None

    for fp in clinical_files:
        if not Path(fp).exists():
            print(f"Warning: clinical file does not exist -> {fp}")
            continue

        df = pd.read_csv(fp).copy()
        df.columns = [str(c).strip() for c in df.columns]

        timepoint = parse_timepoint_from_filename(fp)
        df = strip_timepoint_suffix_from_columns(df, timepoint)

        if id_col not in df.columns:
            raise ValueError(f"{Path(fp).name} does not contain '{id_col}'.")

        df["hospital_patient_id"] = df[id_col].map(extract_digits).astype("Int64")
        df["timepoint"] = timepoint

        cols_keep = ["hospital_patient_id", "timepoint"]
        cols_keep += [c for c in static_cols if c in df.columns]

        if keep_base_vars is None:
            cols_keep += [c for c in df.columns if c not in cols_keep and c != id_col]
        else:
            cols_keep += [c for c in keep_base_vars if c in df.columns]

        tmp = df[cols_keep].dropna(subset=["hospital_patient_id"]).copy()

        if merged is None:
            merged = tmp
        else:
            merged = merged.merge(
                tmp,
                on=["hospital_patient_id", "timepoint"],
                how="outer",
                suffixes=("", "_dup"),
            )

            dup_cols = [c for c in merged.columns if c.endswith("_dup")]
            for dup in dup_cols:
                base = dup[:-4]
                if base not in merged.columns:
                    merged[base] = merged[dup]
                else:
                    merged[base] = merged[base].combine_first(merged[dup])

            merged = merged.drop(columns=dup_cols, errors="ignore")

    if merged is None:
        merged = pd.DataFrame(columns=["hospital_patient_id", "timepoint"] + static_cols)

    return merged


## 1.7. Timepoint assignment

In [28]:
def assign_timepoints_for_patient(
    patient_signals_df: pd.DataFrame,
    patient_visit_df: pd.DataFrame,
) -> pd.DataFrame:
    """
    Attach timepoint and hospital_patient_id to all signal rows of one patient.

    Strategy
    1. If the number of unique acquisition dates matches the number of known visits,
       map dates to PRE / POST / FUP in chronological order.
    2. Otherwise, assign visits by sequential blocks using the expected number of
       acquisitions per visit:
           old -> 9
           new -> 18
    """
    sig = patient_signals_df.copy()

    protocol = sig["protocol_version"].iloc[0]
    expected_per_visit = EXPECTED_ACQUISITIONS_PER_VISIT.get(protocol, np.nan)

    sig = sig.sort_values(
        ["acquisition_datetime", "site_code", "measurement_index_site", "source_file"]
    ).reset_index(drop=True)

    sig["timepoint"] = pd.NA
    sig["hospital_patient_id"] = pd.Series([pd.NA] * len(sig), dtype="Int64")
    sig["assignment_method"] = "unassigned"
    sig["assignment_complete"] = False
    sig["assignment_note"] = pd.NA

    if patient_visit_df.empty:
        sig["assignment_method"] = "no_metadata"
        sig["assignment_note"] = "No elastograph visit metadata available for this patient"
        return sig

    visit_df = patient_visit_df.copy().sort_values("timepoint_order").reset_index(drop=True)

    ordered_timepoints = visit_df["timepoint"].tolist()
    tp_to_hospital = dict(zip(visit_df["timepoint"], visit_df["hospital_patient_id"]))
    n_visits = len(ordered_timepoints)
    n_files = len(sig)

    # -----------------------------------------------------
    # Strategy 1: assign by acquisition date
    # -----------------------------------------------------
    valid_dates = sig["acquisition_date"].dropna()
    unique_dates = sorted(pd.Series(valid_dates.unique()).tolist()) if len(valid_dates) > 0 else []

    if len(unique_dates) == n_visits and n_visits > 0:
        date_to_tp = dict(zip(unique_dates, ordered_timepoints))
        sig["timepoint"] = sig["acquisition_date"].map(date_to_tp)
        sig["hospital_patient_id"] = pd.to_numeric(sig["timepoint"].map(tp_to_hospital), errors="coerce").astype("Int64")
        sig["assignment_method"] = "date_map"
        sig["assignment_complete"] = sig["timepoint"].notna().all()
        sig["assignment_note"] = f"Assigned by {len(unique_dates)} distinct acquisition dates"
        return sig

    # -----------------------------------------------------
    # Strategy 2: assign by sequential visit blocks
    # -----------------------------------------------------
    if pd.notna(expected_per_visit) and expected_per_visit > 0:
        block_index = np.arange(n_files) // int(expected_per_visit)

        assigned_timepoints = []
        for b in block_index:
            if b < n_visits:
                assigned_timepoints.append(ordered_timepoints[int(b)])
            else:
                assigned_timepoints.append(pd.NA)

        sig["timepoint"] = assigned_timepoints
        sig["hospital_patient_id"] = pd.to_numeric(sig["timepoint"].map(tp_to_hospital), errors="coerce").astype("Int64")
        sig["assignment_method"] = "block_map"
        sig["assignment_complete"] = (n_files == n_visits * int(expected_per_visit))
        sig["assignment_note"] = (
            f"Assigned by sequential blocks of {int(expected_per_visit)} files per visit; "
            f"n_files={n_files}, n_visits={n_visits}"
        )
        return sig

    sig["assignment_method"] = "failed"
    sig["assignment_note"] = "Could not assign timepoint"
    return sig

In [29]:
def assign_signal_timepoints(
    signals_raw_df: pd.DataFrame,
    elasto_visit_df: pd.DataFrame,
) -> pd.DataFrame:
    """
    Apply timepoint assignment patient by patient.
    """
    if signals_raw_df.empty:
        return signals_raw_df.copy()

    if elasto_visit_df.empty:
        out = signals_raw_df.copy()
        out["timepoint"] = pd.NA
        out["hospital_patient_id"] = pd.Series([pd.NA] * len(out), dtype="Int64")
        out["assignment_method"] = "no_metadata"
        out["assignment_complete"] = False
        out["assignment_note"] = "No elastograph metadata table available"
        return out

    lookup = {
        key: grp.copy()
        for key, grp in elasto_visit_df.groupby(["protocol_version", "global_patient_id"])
    }

    parts = []
    for key, sig_grp in signals_raw_df.groupby(["protocol_version", "global_patient_id"], sort=False):
        visit_grp = lookup.get(
            key,
            pd.DataFrame(columns=elasto_visit_df.columns)
        )
        parts.append(assign_timepoints_for_patient(sig_grp, visit_grp))

    out = pd.concat(parts, ignore_index=True)
    out["hospital_patient_id"] = pd.to_numeric(out["hospital_patient_id"], errors="coerce").astype("Int64")
    return out

## 1.8. Integration functions

In [30]:
def merge_signal_and_clinical(
    signals_with_visit_df: pd.DataFrame,
    clinical_long_df: pd.DataFrame,
) -> pd.DataFrame:
    """
    Merge signal rows with clinical variables using hospital_patient_id and timepoint.
    """
    if clinical_long_df.empty:
        return signals_with_visit_df.copy()

    out = signals_with_visit_df.merge(
        clinical_long_df,
        on=["hospital_patient_id", "timepoint"],
        how="left",
    )
    return out

## 1.9. Quick QC and visualisation helpers

In [31]:
def check_expected_measurement_counts_raw(signals_raw_df: pd.DataFrame) -> pd.DataFrame:
    """
    Preliminary QC before timepoint assignment.
    This count is aggregated across visits because timepoint is not yet attached.
    """
    qc = (
        signals_raw_df
        .groupby(["protocol_version", "global_patient_id", "site_code"], as_index=False)
        .size()
        .rename(columns={"size": "n_measurements"})
    )

    qc["expected_n_per_site"] = qc["protocol_version"].map(EXPECTED_MEASUREMENTS_PER_SITE)
    qc["count_ok_preliminary"] = qc["n_measurements"] >= qc["expected_n_per_site"]
    return qc

In [32]:
def check_expected_measurement_counts_by_visit(signals_with_visit_df: pd.DataFrame) -> pd.DataFrame:
    """
    QC after timepoint assignment.
    One row per patient x timepoint x site.
    """
    qc = (
        signals_with_visit_df
        .groupby(["protocol_version", "global_patient_id", "timepoint", "site_code"], as_index=False)
        .size()
        .rename(columns={"size": "n_measurements"})
    )

    qc["expected_n_per_site"] = qc["protocol_version"].map(EXPECTED_MEASUREMENTS_PER_SITE)
    qc["count_ok"] = qc["n_measurements"] == qc["expected_n_per_site"]
    return qc

In [33]:
def check_visit_assignment_summary(signals_with_visit_df: pd.DataFrame) -> pd.DataFrame:
    """
    One row per patient x timepoint showing total number of acquisitions.
    """
    qc = (
        signals_with_visit_df
        .groupby(
            ["protocol_version", "global_patient_id", "timepoint", "assignment_method"],
            as_index=False
        )
        .size()
        .rename(columns={"size": "n_acquisitions"})
    )

    qc["expected_n_per_visit"] = qc["protocol_version"].map(EXPECTED_ACQUISITIONS_PER_VISIT)
    qc["visit_count_ok"] = qc["n_acquisitions"] == qc["expected_n_per_visit"]
    return qc

In [34]:
def frequency_coverage_table(signals_long_df: pd.DataFrame) -> pd.DataFrame:
    """
    Cross-tab of how many acquisition files contain each frequency in each protocol.
    """
    if signals_long_df.empty:
        return pd.DataFrame()

    out = (
        signals_long_df
        .groupby(["protocol_version", "frequency_hz"])["signal_record_id"]
        .nunique()
        .reset_index(name="n_files")
        .pivot(index="frequency_hz", columns="protocol_version", values="n_files")
        .fillna(0)
        .astype(int)
        .sort_index()
    )
    return out


## 1.10. Main builder

In [35]:
def build_source_tables(
    signal_roots: List[str],
    old_elasto_csv: Optional[str] = None,
    new_elasto_csv: Optional[str] = None,
    clinical_files: Optional[List[str]] = None,
    keep_clinical_vars: Optional[List[str]] = None,
    build_long_signals: bool = False,
    compute_hash: bool = False,
) -> dict:
    """
    Build the harmonised source tables and attach timepoint to signal files.
    """
    # Signals
    signals_raw_df = build_signal_raw_dataset(
        signal_roots=signal_roots,
        compute_hash=compute_hash,
    )

    # Preliminary QC before timepoint
    raw_measurement_qc_df = check_expected_measurement_counts_raw(signals_raw_df)

    # Elastograph metadata
    elasto_parts = []
    if old_elasto_csv is not None and Path(old_elasto_csv).exists():
        elasto_parts.append(load_elastograph_metadata(old_elasto_csv, protocol_version="old"))
    if new_elasto_csv is not None and Path(new_elasto_csv).exists():
        elasto_parts.append(load_elastograph_metadata(new_elasto_csv, protocol_version="new"))

    elasto_df = pd.concat(elasto_parts, ignore_index=True) if elasto_parts else pd.DataFrame()
    elasto_visit_df = build_elasto_visit_table(elasto_df)

    # Attach timepoint to signal rows
    signals_with_visit_df = assign_signal_timepoints(signals_raw_df, elasto_visit_df)

    # Clinical
    clinical_long_df = pd.DataFrame()
    if clinical_files:
        clinical_long_df = load_clinical_files_to_long(
            clinical_files=clinical_files,
            id_col="ID",
            static_cols=["Sex", "Age"],
            keep_base_vars=keep_clinical_vars,
        )

    signals_clinical_df = merge_signal_and_clinical(
        signals_with_visit_df=signals_with_visit_df,
        clinical_long_df=clinical_long_df,
    )

    # QC after timepoint assignment
    site_visit_qc_df = check_expected_measurement_counts_by_visit(signals_with_visit_df)
    visit_assignment_qc_df = check_visit_assignment_summary(signals_with_visit_df)

    # Optional long-format table
    signals_long_df = pd.DataFrame()
    if build_long_signals:
        signals_long_df = build_signal_long_dataset_light(signals_with_visit_df)

    return {
        "signals_raw_df": signals_raw_df,
        "signals_with_visit_df": signals_with_visit_df,
        "signals_long_df": signals_long_df,
        "elasto_df": elasto_df,
        "elasto_visit_df": elasto_visit_df,
        "clinical_long_df": clinical_long_df,
        "signals_clinical_df": signals_clinical_df,
        "raw_measurement_qc_df": raw_measurement_qc_df,
        "site_visit_qc_df": site_visit_qc_df,
        "visit_assignment_qc_df": visit_assignment_qc_df,
    }

## 1.11. Paths

In [36]:
SIGNAL_ROOTS = [
    "/content/drive/MyDrive/PhD/oldcode/",
    "/content/drive/MyDrive/PhD/newcode/",
]

OLD_ELASTO_CSV = "/content/drive/MyDrive/PhD/files_oldcode/measured_with_elastograph_patients_old.csv"
NEW_ELASTO_CSV = "/content/drive/MyDrive/PhD/files_newcode/measured_with_elastograph_patients_new.csv"

CLINICAL_FILES = [
    "/content/drive/MyDrive/PhD/files_oldcode/BodyComposition_PRE.csv",
    "/content/drive/MyDrive/PhD/files_oldcode/BodyComposition_POST.csv",
    "/content/drive/MyDrive/PhD/files_oldcode/BodyComposition_FUP.csv",
    "/content/drive/MyDrive/PhD/files_oldcode/BloodComposition_PRE.csv",
    "/content/drive/MyDrive/PhD/files_oldcode/BloodComposition_POST.csv",
    "/content/drive/MyDrive/PhD/files_oldcode/BloodComposition_FUP.csv",
]

KEEP_CLINICAL_VARS = [
    "BC_WC_mean",
    "BLOB_TG",
    "BLOB_HDL",
    "BP_SBP_ave",
    "BP_DBP_ave",
    "BLOB_Gluc",
]

## 1.12. Build tables

In [37]:
t0 = time.time()

tables = build_source_tables(
    signal_roots=SIGNAL_ROOTS,
    old_elasto_csv=OLD_ELASTO_CSV,
    new_elasto_csv=NEW_ELASTO_CSV,
    clinical_files=CLINICAL_FILES,
    keep_clinical_vars=KEEP_CLINICAL_VARS,
    build_long_signals=False,   # set True if you want the long table directly here
    compute_hash=False,         # keep False unless you really need file hashing
)

t1 = time.time()
print(f"\nFull source-table build finished in {t1 - t0:.2f} s")


signals_raw_df = tables["signals_raw_df"]
signals_with_visit_df = tables["signals_with_visit_df"]
elasto_df = tables["elasto_df"]
elasto_visit_df = tables["elasto_visit_df"]
clinical_long_df = tables["clinical_long_df"]
signals_clinical_df = tables["signals_clinical_df"]
raw_measurement_qc_df = tables["raw_measurement_qc_df"]
site_visit_qc_df = tables["site_visit_qc_df"]
visit_assignment_qc_df = tables["visit_assignment_qc_df"]

# Build the light long-format signal table after timepoint assignment
t2 = time.time()
signals_long_df = build_signal_long_dataset_light(signals_with_visit_df)
t3 = time.time()
print(f"Light long-format signal table built in {t3 - t2:.2f} s")


Signal root /content/drive/MyDrive/PhD/oldcode/ -> 1914 files found
Signal root /content/drive/MyDrive/PhD/newcode/ -> 1581 files found
Signal files found in total: 3495
Parsing file 1/3495: in_test_0001_A_Study_angle_1_rep_1_23_03_25_09_48_52.csv
Parsing file 50/3495: in_test_0003_B_Study_angle_2_rep_2_23_03_25_10_47_19.csv
Parsing file 100/3495: in_test_0006_A_Study_angle_2_rep_2_23_03_25_12_23_40.csv
Parsing file 150/3495: in_test_0008_P_Study_angle_3_rep_2_11_05_25_09_58_13.csv
Parsing file 200/3495: in_test_0011_P_Study_angle_1_rep_2_11_05_25_10_48_55.csv
Parsing file 250/3495: in_test_0014_B_Study_angle_2_rep_2_11_05_25_11_57_34.csv
Parsing file 300/3495: in_test_0017_A_Study_angle_3_rep_2_11_05_25_13_00_02.csv
Parsing file 350/3495: in_test_0020_A_Study_angle_1_rep_2_18_05_25_09_12_24.csv
Parsing file 400/3495: in_test_0022_P_Study_angle_2_rep_1_18_05_25_10_04_17.csv
Parsing file 450/3495: in_test_0025_B_Study_angle_3_rep_1_18_05_25_11_34_16.csv
Parsing file 500/3495: in_test_00

## 1.13. Visual inspection

In [38]:
print("\nSignals raw dataset")
display(
    signals_raw_df[
        [
            "signal_record_id",
            "protocol_version",
            "global_patient_id",
            "site_code",
            "measurement_index_site",
            "acquisition_datetime",
            "source_file",
            "n_frequency_channels",
            "available_freqs_hz",
            "parse_ok",
        ]
    ].head(10)
)


Signals raw dataset


,signal_record_id,protocol_version,global_patient_id,site_code,measurement_index_site,acquisition_datetime,source_file,n_frequency_channels,available_freqs_hz,parse_ok
0,sig_000000,new,159,A,1,2025-03-23 09:48:52,in_test_0001_A_Study_angle_1_rep_1_23_03_25_09...,9,"[80, 100, 200, 400, 600, 800, 1000, 1500, 2000]",True
1,sig_000001,new,159,A,2,2025-03-23 09:49:10,in_test_0001_A_Study_angle_1_rep_2_23_03_25_09...,9,"[80, 100, 200, 400, 600, 800, 1000, 1500, 2000]",True
2,sig_000002,new,159,A,3,2025-03-23 09:49:26,in_test_0001_A_Study_angle_2_rep_1_23_03_25_09...,9,"[80, 100, 200, 400, 600, 800, 1000, 1500, 2000]",True
3,sig_000003,new,159,A,4,2025-03-23 09:49:42,in_test_0001_A_Study_angle_2_rep_2_23_03_25_09...,9,"[80, 100, 200, 400, 600, 800, 1000, 1500, 2000]",True
4,sig_000004,new,159,A,5,2025-03-23 09:49:57,in_test_0001_A_Study_angle_3_rep_1_23_03_25_09...,9,"[80, 100, 200, 400, 600, 800, 1000, 1500, 2000]",True
5,sig_000005,new,159,A,6,2025-03-23 09:50:11,in_test_0001_A_Study_angle_3_rep_2_23_03_25_09...,9,"[80, 100, 200, 400, 600, 800, 1000, 1500, 2000]",True
6,sig_000006,new,159,B,1,2025-03-23 09:43:08,in_test_0001_B_Study_angle_1_rep_1_23_03_25_09...,9,"[80, 100, 200, 400, 600, 800, 1000, 1500, 2000]",True
7,sig_000007,new,159,B,2,2025-03-23 09:43:24,in_test_0001_B_Study_angle_1_rep_2_23_03_25_09...,9,"[80, 100, 200, 400, 600, 800, 1000, 1500, 2000]",True
8,sig_000008,new,159,B,3,2025-03-23 09:43:38,in_test_0001_B_Study_angle_2_rep_1_23_03_25_09...,9,"[80, 100, 200, 400, 600, 800, 1000, 1500, 2000]",True
9,sig_000009,new,159,B,4,2025-03-23 09:43:53,in_test_0001_B_Study_angle_2_rep_2_23_03_25_09...,9,"[80, 100, 200, 400, 600, 800, 1000, 1500, 2000]",True


In [39]:
print("\nSignals with assigned timepoint")
display(
    signals_with_visit_df[
        [
            "signal_record_id",
            "protocol_version",
            "global_patient_id",
            "hospital_patient_id",
            "timepoint",
            "site_code",
            "measurement_index_site",
            "assignment_method",
            "assignment_complete",
            "source_file",
        ]
    ].head(20)
)


Signals with assigned timepoint


,signal_record_id,protocol_version,global_patient_id,hospital_patient_id,timepoint,site_code,measurement_index_site,assignment_method,assignment_complete,source_file
0,sig_000006,new,159,579,FUP,B,1,date_map,True,in_test_0001_B_Study_angle_1_rep_1_23_03_25_09...
1,sig_000007,new,159,579,FUP,B,2,date_map,True,in_test_0001_B_Study_angle_1_rep_2_23_03_25_09...
2,sig_000008,new,159,579,FUP,B,3,date_map,True,in_test_0001_B_Study_angle_2_rep_1_23_03_25_09...
3,sig_000009,new,159,579,FUP,B,4,date_map,True,in_test_0001_B_Study_angle_2_rep_2_23_03_25_09...
4,sig_000010,new,159,579,FUP,B,5,date_map,True,in_test_0001_B_Study_angle_3_rep_1_23_03_25_09...
5,sig_000011,new,159,579,FUP,B,6,date_map,True,in_test_0001_B_Study_angle_3_rep_2_23_03_25_09...
6,sig_000012,new,159,579,FUP,P,1,date_map,True,in_test_0001_P_Study_angle_1_rep_1_23_03_25_09...
7,sig_000013,new,159,579,FUP,P,2,date_map,True,in_test_0001_P_Study_angle_1_rep_2_23_03_25_09...
8,sig_000014,new,159,579,FUP,P,3,date_map,True,in_test_0001_P_Study_angle_2_rep_1_23_03_25_09...
9,sig_000015,new,159,579,FUP,P,4,date_map,True,in_test_0001_P_Study_angle_2_rep_2_23_03_25_09...


In [40]:
print("\nNumber of acquisition files by protocol")
display(
    signals_raw_df["protocol_version"]
    .value_counts(dropna=False)
    .rename_axis("protocol_version")
    .reset_index(name="n_files")
)


Number of acquisition files by protocol


,protocol_version,n_files
0,old,1914
1,new,1581


In [41]:
print("\nNumber of acquisition files by protocol and site")
display(
    signals_raw_df
    .groupby(["protocol_version", "site_code"])
    .size()
    .reset_index(name="n_files")
    .sort_values(["protocol_version", "site_code"])
)


Number of acquisition files by protocol and site


,protocol_version,site_code,n_files
0,new,A,518
1,new,B,541
2,new,P,522
3,old,A,727
4,old,B,566
5,old,P,621


In [42]:
print("\nUnique patients by protocol")
display(
    signals_raw_df
    .groupby("protocol_version")["global_patient_id"]
    .nunique()
    .reset_index(name="n_unique_patients")
)


Unique patients by protocol


,protocol_version,n_unique_patients
0,new,86
1,old,157


In [43]:
print("\nPreliminary raw QC by patient and site")
display(
    raw_measurement_qc_df
    .sort_values(["protocol_version", "global_patient_id", "site_code"])
    .head(30)
)


Preliminary raw QC by patient and site


,protocol_version,global_patient_id,site_code,n_measurements,expected_n_per_site,count_ok_preliminary
0,new,159,A,6,6,True
1,new,159,B,6,6,True
2,new,159,P,6,6,True
3,new,160,A,6,6,True
4,new,160,B,6,6,True
5,new,160,P,6,6,True
6,new,161,A,6,6,True
7,new,161,B,12,6,True
8,new,161,P,6,6,True
9,new,162,A,6,6,True


In [44]:
print("\nVisit assignment summary")
display(
    visit_assignment_qc_df
    .sort_values(["protocol_version", "global_patient_id", "timepoint"])
    .head(30)
)


Visit assignment summary


,protocol_version,global_patient_id,timepoint,assignment_method,n_acquisitions,expected_n_per_visit,visit_count_ok
0,new,159,FUP,date_map,18,18,True
1,new,160,FUP,date_map,18,18,True
2,new,161,FUP,date_map,24,18,False
3,new,162,FUP,date_map,18,18,True
4,new,163,FUP,date_map,18,18,True
5,new,164,FUP,date_map,18,18,True
6,new,165,FUP,date_map,18,18,True
7,new,166,FUP,date_map,18,18,True
8,new,167,FUP,date_map,18,18,True
9,new,168,FUP,date_map,18,18,True


In [45]:
print("\nQC by patient, timepoint and site")
display(
    site_visit_qc_df
    .sort_values(["protocol_version", "global_patient_id", "timepoint", "site_code"])
    .head(30)
)


QC by patient, timepoint and site


,protocol_version,global_patient_id,timepoint,site_code,n_measurements,expected_n_per_site,count_ok
0,new,159,FUP,A,6,6,True
1,new,159,FUP,B,6,6,True
2,new,159,FUP,P,6,6,True
3,new,160,FUP,A,6,6,True
4,new,160,FUP,B,6,6,True
5,new,160,FUP,P,6,6,True
6,new,161,FUP,A,6,6,True
7,new,161,FUP,B,12,6,False
8,new,161,FUP,P,6,6,True
9,new,162,FUP,A,6,6,True


In [46]:
print("\nFrequency coverage by protocol")
display(frequency_coverage_table(signals_long_df))


Frequency coverage by protocol


protocol_version,new,old
frequency_hz,,
80,1581,0
100,1581,0
200,1581,0
400,1581,1914
500,0,1914
600,1581,1914
700,0,1914
800,1581,1914
900,0,1914


In [47]:
if not elasto_df.empty:
    print("\nElastograph raw metadata")
    display(elasto_df.head())

    print("\nElastograph visit metadata")
    display(elasto_visit_df.head())

    print("\nMetadata counts by protocol and timepoint")
    display(
        elasto_visit_df
        .groupby(["protocol_version", "timepoint"])
        .size()
        .reset_index(name="n_rows")
        .sort_values(["protocol_version", "timepoint"])
    )


Elastograph raw metadata


,protocol_version,global_patient_id,hospital_patient_id,timepoint
0,old,1,540,POST
1,old,2,533,POST
2,old,3,534,POST
3,old,4,535,POST
4,old,5,538,POST



Elastograph visit metadata


,protocol_version,global_patient_id,timepoint,hospital_patient_id,n_elasto_rows,timepoint_order
0,new,159,FUP,579,1,2
1,new,160,FUP,578,1,2
2,new,161,FUP,586,1,2
3,new,162,FUP,584,1,2
4,new,163,FUP,582,1,2



Metadata counts by protocol and timepoint


,protocol_version,timepoint,n_rows
0,new,FUP,86
1,old,FUP,20
2,old,POST,96
3,old,PRE,41


In [48]:
if not clinical_long_df.empty:
    print("\nClinical long table")
    display(clinical_long_df.head())

    print("\nClinical rows by timepoint")
    display(
        clinical_long_df["timepoint"]
        .value_counts(dropna=False)
        .rename_axis("timepoint")
        .reset_index(name="n_rows")
    )


Clinical long table


,hospital_patient_id,timepoint,Sex,Age,BC_WC_mean,BLOB_TG,BLOB_HDL,BP_SBP_ave,BP_DBP_ave,BLOB_Gluc
0,500,FUP,1.0,60.0,100.1,114.0,51.0,113.5,72.5,88.0
1,500,POST,1.0,60.0,97.9,106.0,55.0,99.0,68.0,89.0
2,500,PRE,1.0,60.0,101.2,107.0,57.0,104.0,68.0,96.0
3,501,FUP,1.0,43.0,102.0,120.0,54.0,95.5,72.5,73.0
4,501,POST,1.0,43.0,101.3,64.0,62.0,90.0,64.5,81.0



Clinical rows by timepoint


,timepoint,n_rows
0,FUP,187
1,POST,187
2,PRE,187


In [49]:
print("\nMerged signal + clinical table")
display(
    signals_clinical_df[
        [
            "signal_record_id",
            "protocol_version",
            "global_patient_id",
            "hospital_patient_id",
            "timepoint",
            "site_code",
            "measurement_index_site",
            "assignment_method",
        ] + [c for c in KEEP_CLINICAL_VARS if c in signals_clinical_df.columns]
    ].head(20)
)


Merged signal + clinical table


,signal_record_id,protocol_version,global_patient_id,hospital_patient_id,timepoint,site_code,measurement_index_site,assignment_method,BC_WC_mean,BLOB_TG,BLOB_HDL,BP_SBP_ave,BP_DBP_ave,BLOB_Gluc
0,sig_000006,new,159,579,FUP,B,1,date_map,106.5,156.0,32.0,109.5,80.0,103.0
1,sig_000007,new,159,579,FUP,B,2,date_map,106.5,156.0,32.0,109.5,80.0,103.0
2,sig_000008,new,159,579,FUP,B,3,date_map,106.5,156.0,32.0,109.5,80.0,103.0
3,sig_000009,new,159,579,FUP,B,4,date_map,106.5,156.0,32.0,109.5,80.0,103.0
4,sig_000010,new,159,579,FUP,B,5,date_map,106.5,156.0,32.0,109.5,80.0,103.0
5,sig_000011,new,159,579,FUP,B,6,date_map,106.5,156.0,32.0,109.5,80.0,103.0
6,sig_000012,new,159,579,FUP,P,1,date_map,106.5,156.0,32.0,109.5,80.0,103.0
7,sig_000013,new,159,579,FUP,P,2,date_map,106.5,156.0,32.0,109.5,80.0,103.0
8,sig_000014,new,159,579,FUP,P,3,date_map,106.5,156.0,32.0,109.5,80.0,103.0
9,sig_000015,new,159,579,FUP,P,4,date_map,106.5,156.0,32.0,109.5,80.0,103.0
